In [47]:
class IATGAggregator:
    def __init__(self, inventory):
        """
        inventory: dict of HD: weight (e.g., {"1.2": 50, "1.3.1": 100})
        """
        # Standardizing all variants into their primary IATG aggregation buckets
        self.raw_inv = inventory
        self.weights = {
            "1.1": inventory.get("1.1", 0) + inventory.get("1.5", 0),
            # IATG Rule: HD 1.2 (unspecified) and 1.2.3 are treated as 1.2.1
            "1.2.1": (inventory.get("1.2.1", 0) + 
                      inventory.get("1.2.3", 0) + 
                      inventory.get("1.2", 0)),
            "1.2.2": inventory.get("1.2.2", 0),
            "1.3": (inventory.get("1.3", 0) + 
                    inventory.get("1.3.1", 0) + 
                    inventory.get("1.3.2", 0)),
            "1.4": inventory.get("1.4", 0),
            "1.6": inventory.get("1.6", 0)
        }

    def calculate_aggregation(self):
        print("--- IATG Step-by-Step Aggregation Trace ---")
        for hd, weight in self.raw_inv.items():
            if weight > 0:
                print(f"Input: {hd} = {weight} kg")
        print("-" * 40)

        # 1. The 1.1/1.5 Rule
        # Aggregates everything EXCEPT 1.4
        if self.weights["1.1"] > 0:
            total = sum(self.weights.values()) - self.weights["1.4"]
            print(f"[RULE 1] HD 1.1/1.5 detected.")
            print(f"         Aggregating ALL items to 1.1 except 1.4 which is ignored: {total} kg")
            if self.weights["1.4"] > 0:
                print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.1", total

        # 2. The 1.2.1 Rule (Includes general 1.2 and 1.2.3)
        if self.weights["1.2.1"] > 0:
            # Aggregates everything EXCEPT 1.4
            total = (self.weights["1.2.1"] + self.weights["1.2.2"] + 
                     self.weights["1.3"] + self.weights["1.6"])
            print(f"[RULE 2] HD 1.2.1 (or generic 1.2) detected.")
            print(f"         Aggregating 1.2.x, 1.3, and 1.6 as 1.2.1: {total} kg")
            if self.weights["1.4"] > 0:
                print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")            
            return "1.2.1", total

        # 3. The 1.2.2 Rule
        if self.weights["1.2.2"] > 0:
            total = self.weights["1.2.2"] + self.weights["1.3"] + self.weights["1.6"]
            print(f"[RULE 3] HD 1.2.2 detected.")
            print(f"         Aggregating 1.2.2, 1.3, and 1.6 as 1.2.2: {total} kg")
            if self.weights["1.4"] > 0:
                print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.2.2", total

        # 4. The 1.3 Rule
        if self.weights["1.3"] > 0:
            total = self.weights["1.3"] + self.weights["1.6"]
            print(f"[RULE 4] HD 1.3 detected.")
            print(f"         Aggregating 1.3 and 1.6 as 1.3: {total} kg")
            if self.weights["1.4"] > 0:
                print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.3", total

        # 5. The 1.6 Rule
        if self.weights["1.6"] > 0:
            print(f"[RULE 5] Only HD 1.6 and 1.4 present. 1.4 is ignored")
            if self.weights["1.4"] > 0:
                print(f"         (HD 1.4 of {self.weights['1.4']} kg is ignored per IATG)")
            return "1.6", self.weights["1.6"]

        # 6. The 1.4 Rule
        print(f"[RULE 6] Only HD 1.4")
        return "1.4", self.weights["1.4"]

# --- Example  ---
mixed_storage = {

"1.2.1": 5907.0535,
"1.2": 626.0703,
"1.3": 43.4082,
"1.2.2": 0.2532,
"1.4": 0.1640


}

aggregator = IATGAggregator(mixed_storage)
final_hd, final_new = aggregator.calculate_aggregation()

print("-" * 40)
print(f"FINAL AGGREGATED HD: {final_hd}")
print(f"FINAL AGGREGATED NEW: {final_new} kg")

--- IATG Step-by-Step Aggregation Trace ---
Input: 1.2.1 = 5907.0535 kg
Input: 1.2 = 626.0703 kg
Input: 1.3 = 43.4082 kg
Input: 1.2.2 = 0.2532 kg
Input: 1.4 = 0.164 kg
----------------------------------------
[RULE 2] HD 1.2.1 (or generic 1.2) detected.
         Aggregating 1.2.x, 1.3, and 1.6 as 1.2.1: 6576.7852 kg
         (HD 1.4 of 0.164 kg is ignored per IATG)
----------------------------------------
FINAL AGGREGATED HD: 1.2.1
FINAL AGGREGATED NEW: 6576.7852 kg
